# Évaluation V5 - Dataset de Test Complet (3775 images)

Évaluation du modèle final V5 sur le dataset de test officiel non étiqueté

In [1]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================
import os
import json
import re
import random
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import Counter

from unsloth import FastVisionModel
from peft import PeftModel
from transformers import AutoProcessor

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch: 2.9.0+cu128
CUDA: True


In [12]:
# ============================================================
# CELL 2: CONFIGURATION
# ============================================================
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
CHECKPOINT_PATH = BASE_DIR / "checkpoints" / "stage2_v5_unified_final" / "checkpoint-2100"

# Dataset de test - 3775 images
TEST_IMAGES_DIR = "/home/hounfodjidagba/learning/cassava/data/cassava/test/0/"
TEST_LABELS_CSV = "/home/hounfodjidagba/learning/cassava/data/sample_submission_file.csv"

MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)
CLASS_NAMES = ["CBB", "CBSD", "CGM", "CMD", "Healthy"]

# Mapping labels CSV vers classes modèle
LABEL_MAPPING = {
    'cbb': 'CBB',
    'cbsd': 'CBSD', 
    'cgm': 'CGM',
    'cmd': 'CMD',
    'healthy': 'Healthy'
}

print(f"Labels CSV: {TEST_LABELS_CSV}")

# Stage 1 results for comparisonprint(f"Dataset de test: {TEST_IMAGES_DIR}")

STAGE1_RESULTS = {
    "global": 80.52,
    "CBB": 55.13,
    "CBSD": 83.99,
    "CGM": 56.47,
    "CMD": 85.98,
    "Healthy": 75.37
}

Labels CSV: /home/hounfodjidagba/learning/cassava/data/sample_submission_file.csv


In [13]:
# ============================================================
# CELL 3: CHARGER MODÈLE DEPUIS CHECKPOINT
# ============================================================
print("Chargement du modèle de base...")
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
)

print(f"Chargement du checkpoint V5: {CHECKPOINT_PATH}")
model = PeftModel.from_pretrained(model, str(CHECKPOINT_PATH))

processor = AutoProcessor.from_pretrained(str(CHECKPOINT_PATH))

# Mode inférence
FastVisionModel.for_inference(model)
print("✅ Modèle chargé en mode inférence")

Chargement du modèle de base...
==((====))==  Unsloth 2026.1.1: Fast Qwen2_5_Vl patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Chargement du checkpoint V5: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v5_unified_final/checkpoint-2100
✅ Modèle chargé en mode inférence


In [14]:
# ============================================================
# CELL 4: CHARGER DATASET DE TEST
# ============================================================
print("Chargement du dataset de test...")

# Charger les labels depuis le CSV
test_df = pd.read_csv(TEST_LABELS_CSV)
print(f"Test samples dans CSV: {len(test_df)}")

# Créer la liste des échantillons de test
test_data = []
valid_samples = 0
missing_images = 0

for _, row in test_df.iterrows():
    image_path = os.path.join(TEST_IMAGES_DIR, row['Id'])
    if os.path.exists(image_path):
        # Mapper le label CSV vers la classe modèle
        csv_label = row['Category'].lower()
        model_class = LABEL_MAPPING.get(csv_label, "Unknown")
        
        if model_class != "Unknown":
            test_data.append({
                'image_path': image_path,
                'image_id': row['Id'],
                'csv_label': csv_label,
                'actual': model_class
            })
            valid_samples += 1
        else:
            print(f"⚠️ Label inconnu: {csv_label} pour {row['Id']}")
    else:
        missing_images += 1


print(f"✅ Échantillons de test valides: {valid_samples}")
if missing_images > 0:
    print(f"⚠️ Images manquantes: {missing_images}")

# Distribution du dataset de test
test_dist = Counter(sample['actual'] for sample in test_data)
print("\nDistribution du dataset de test:")
for cls in CLASS_NAMES:
    print(f"  {cls}: {test_dist[cls]}")

Chargement du dataset de test...
Test samples dans CSV: 3774
✅ Échantillons de test valides: 3774

Distribution du dataset de test:
  CBB: 753
  CBSD: 731
  CGM: 706
  CMD: 800
  Healthy: 784


In [15]:
# ============================================================
# CELL 5: FONCTIONS D'ÉVALUATION
# ============================================================
def generate_response(image_path: str, question: str, max_new_tokens: int = 256):
    """Génère une réponse pour une image."""
    try:
        image = Image.open(image_path).convert('RGB').resize(TARGET_IMAGE_SIZE)
    except:
        return "Erreur"

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question}
        ]
    }]

    input_text = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = processor(
        text=[input_text], images=[[image]],
        return_tensors="pt", padding=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.lower().split("assistant")[-1].strip()
    return response

def extract_disease(response: str) -> str:
    """Extrait la maladie de la réponse."""
    response_lower = response.lower()
    
    # Chercher "diagnostic :" d'abord (format V5)
    if "diagnostic" in response_lower:
        match = re.search(r'diagnostic\s*:\s*([\w\s()]+?)[\.\,]', response_lower)
        if match:
            diag = match.group(1).strip()
            if "cmd" in diag or "mosaic" in diag:
                return "CMD"
            elif "cbsd" in diag or "brown streak" in diag:
                return "CBSD"
            elif "cbb" in diag or "bacterial" in diag:
                return "CBB"
            elif "cgm" in diag or "green" in diag:
                return "CGM"
            elif "healthy" in diag or "sain" in diag:
                return "Healthy"
    
    # Chercher "disease:" (format Stage 1)
    if "disease:" in response_lower:
        match = re.search(r'disease:\s*([\w\s()]+)', response_lower)
        if match:
            diag = match.group(1).strip()
            if "cmd" in diag or "mosaic" in diag:
                return "CMD"
            elif "cbsd" in diag or "brown streak" in diag:
                return "CBSD"
            elif "cbb" in diag or "bacterial" in diag:
                return "CBB"
            elif "cgm" in diag or "green" in diag:
                return "CGM"
            elif "healthy" in diag:
                return "Healthy"
    
    # Fallback - chercher les mots-clés
    if "cmd" in response_lower or "mosaic" in response_lower:
        return "CMD"
    elif "cbsd" in response_lower or "brown streak" in response_lower:
        return "CBSD"
    elif "cbb" in response_lower or "bacterial" in response_lower:
        return "CBB"
    elif "cgm" in response_lower or "green mottle" in response_lower or "green mite" in response_lower:
        return "CGM"
    elif "healthy" in response_lower or "sain" in response_lower:
        return "Healthy"
    
    return "Unknown"

print("✅ Fonctions d'évaluation prêtes")

✅ Fonctions d'évaluation prêtes


In [16]:
# ============================================================
# CELL 6: ÉVALUATION CLASSIFICATION
# ============================================================
print("="*60)
print("ÉVALUATION V5 - DATASET DE TEST COMPLET")
print("="*60)

# Questions pour l'évaluation
questions = [
    "What cassava disease is shown in this image?",
    "Quelle maladie affecte cette feuille de manioc ?"
]

print(f"\n📊 Évaluation sur {len(test_data)} images de test...")
print("⏱️ Temps estimé: ~45-60 minutes")

# Optionnel: sous-échantillonner pour test rapide (décommenter pour test)
print("🔬 Mode test rapide: 100 échantillons")
random.seed(42)
test_data_eval = random.sample(test_data, 100)
# Utiliser test_data pour l'évaluation complète
# test_data_eval = test_data

results = []
for i, sample in enumerate(tqdm(test_data_eval, desc="Evaluating")):
    question = random.choice(questions)
    response = generate_response(sample['image_path'], question)
    predicted = extract_disease(response)
    
    results.append({
        'image_id': sample['image_id'],
        'actual': sample['actual'],
        'predicted': predicted,
        'correct': predicted == sample['actual'],
        'response': response[:300]
    })
    
    # Affichage du progrès tous les 100 échantillons
    if (i + 1) % 100 == 0:
        current_acc = sum(1 for r in results if r['correct']) / len(results) * 100
        print(f"\n📊 Progrès {i+1}/{len(test_data_eval)}: Accuracy actuelle = {current_acc:.1f}%")


# Calculer accuracy
correct = sum(1 for r in results if r['correct'])
accuracy = correct / len(results) * 100

print(f"\n" + "="*60)
print(f"RÉSULTATS V5 - DATASET DE TEST COMPLET")
print(f"="*60)
print(f"\n📊 Accuracy sur dataset de test:")
print(f"  - Échantillons évalués: {len(results):,}")
print(f"  - Prédictions correctes: {correct:,}")
print(f"  - Accuracy globale: {accuracy:.2f}%")

print(f"\n📁 Informations dataset:")
print(f"  - Source: /data/cassava/test/0/")
print(f"  - Labels: sample_submission_file.csv")
print(f"  - Total images test: {len(test_data):,}")

# Comparaison avec Stage 1 (pour référence)
stage1_global = STAGE1_RESULTS.get('global', 0)
if stage1_global > 0:
    diff = accuracy - stage1_global
    print(f"\n🔄 Comparaison avec Stage 1 (référence):")
    print(f"  - Stage 1 (validation): {stage1_global:.1f}%")
    print(f"  - V5 (test): {accuracy:.1f}%")
    print(f"  - Différence: {diff:+.1f}%")
    print(f"  ⚠️ Note: Comparaison indicative (datasets différents)")

# Par classe - métriques détaillées
print(f"\n📈 Métriques par classe:")
class_metrics = {}
for cls in CLASS_NAMES:
    cls_samples = [r for r in results if r['actual'] == cls]
    if cls_samples:
        cls_correct = sum(1 for r in cls_samples if r['correct'])
        cls_acc = cls_correct / len(cls_samples) * 100
        class_metrics[cls] = {
            'accuracy': cls_acc,
            'correct': cls_correct,
            'total': len(cls_samples)
        }
        
        # Statut basé sur la performance
        if cls_acc >= 80:
            status = "🎆"  # Excellent
        elif cls_acc >= 70:
            status = "✅"  # Bon
        elif cls_acc >= 60:
            status = "⚠️"   # Moyen
        else:
            status = "❌"  # Faible
            
        print(f"  {status} {cls}: {cls_acc:.1f}% ({cls_correct}/{len(cls_samples)})")
        
        # Comparison avec Stage 1 pour référence
        stage1_acc = STAGE1_RESULTS.get(cls, 0)
        if stage1_acc > 0:
            cls_diff = cls_acc - stage1_acc
            print(f"      vs Stage 1: {stage1_acc:.1f}% (Δ{cls_diff:+.1f}%)")
    else:
        print(f"  ⚠️ {cls}: Aucun échantillon trouvé")
        class_metrics[cls] = {'accuracy': 0, 'correct': 0, 'total': 0}

ÉVALUATION V5 - DATASET DE TEST COMPLET

📊 Évaluation sur 3774 images de test...
⏱️ Temps estimé: ~45-60 minutes
🔬 Mode test rapide: 100 échantillons


Evaluating: 100%|██████████| 100/100 [03:05<00:00,  1.85s/it]


📊 Progrès 100/100: Accuracy actuelle = 24.0%

RÉSULTATS V5 - DATASET DE TEST COMPLET

📊 Accuracy sur dataset de test:
  - Échantillons évalués: 100
  - Prédictions correctes: 24
  - Accuracy globale: 24.00%

📁 Informations dataset:
  - Source: /data/cassava/test/0/
  - Labels: sample_submission_file.csv
  - Total images test: 3,774

🔄 Comparaison avec Stage 1 (référence):
  - Stage 1 (validation): 80.5%
  - V5 (test): 24.0%
  - Différence: -56.5%
  ⚠️ Note: Comparaison indicative (datasets différents)

📈 Métriques par classe:
  ❌ CBB: 4.5% (1/22)
      vs Stage 1: 55.1% (Δ-50.6%)
  ❌ CBSD: 42.9% (9/21)
      vs Stage 1: 84.0% (Δ-41.1%)
  ❌ CGM: 16.7% (2/12)
      vs Stage 1: 56.5% (Δ-39.8%)
  ❌ CMD: 57.1% (12/21)
      vs Stage 1: 86.0% (Δ-28.8%)
  ❌ Healthy: 0.0% (0/24)
      vs Stage 1: 75.4% (Δ-75.4%)


In [8]:
# ============================================================
# CELL 7: DIAGNOSTIC - RÉPONSES BRUTES
# ============================================================
print("="*60)
print("DIAGNOSTIC: RÉPONSES BRUTES")
print("="*60)

# 10 samples pour diagnostic
diagnostic_samples = random.sample(results, min(10, len(results)))

for i, sample in enumerate(diagnostic_samples):
    print(f"\n{'='*60}")
    print(f"[{i+1}] Classe réelle: {sample['actual']}")
    print(f"    Classe prédite: {sample['predicted']}")
    print(f"    Correct: {'✅' if sample['correct'] else '❌'}")
    print(f"{'='*60}")
    print(f"RÉPONSE:")
    print(f"{sample['response']}")
    if len(sample['response']) >= 300:
        print("...")

DIAGNOSTIC: RÉPONSES BRUTES

[1] Classe réelle: CBB
    Classe prédite: CMD
    Correct: ❌
RÉPONSE:
d'après l'image, votre manioc est atteint de la **mosaïque du manioc** (cmd). j'observe : déformation possible des folioles, nanisme dans les cas avancés, motifs mosaïque jaune-vert sur les feuilles. c'est un stade général (sévérité non spécifiée).

[2] Classe réelle: CBSD
    Classe prédite: CBSD
    Correct: ✅
RÉPONSE:
diagnostic : **cbsd** - striure brune du manioc. sévérité : général. symptômes observés : nécrose brune interne des racines, chlorose sur feuilles matures (pas les jeunes), stries brunes possibles sur tiges.

[3] Classe réelle: CBSD
    Classe prédite: CGM
    Correct: ❌
RÉPONSE:
d'après l'image, votre manioc est atteint de la **acarien vert du manioc** (cgm). j'observe : aspect moucheté sur face supérieure des feuilles, folioles petites, non développées, étroites, extension de la chlorose (bronzage). c'est un stade dégâts d'acariens.

[4] Classe réelle: CGM
    Classe p

In [9]:
# ============================================================
# CELL 8: TEST VQA
# ============================================================
print("="*60)
print("TEST VQA - Questions variées")
print("="*60)

vqa_questions = [
    "Quels sont les signes de maladie que vous observez ?",
    "Que dois-je faire pour traiter cette maladie ?",
    "Est-ce un problème de maladie ou d'insectes ?",
    "Qu'est-ce qui cause ces symptômes ?"
]

vqa_samples = random.sample(test_data, min(10, len(test_data)))

for i, sample in enumerate(vqa_samples):
    question = random.choice(vqa_questions)
    response = generate_response(sample['image_path'], question)
    
    print(f"\n{'='*60}")
    print(f"[{i+1}] Classe: {sample['actual']}")
    print(f"    Question: {question}")
    print(f"    Réponse: {response[:400]}")
    if len(response) > 400:
        print("...")

TEST VQA - Questions variées



[1] Classe: CMD
    Question: Qu'est-ce qui cause ces symptômes ?
    Réponse: ces symptômes sont causés par begomovirus (acmv, eacmv), responsable de la mosaïque du manioc. la transmission se fait par mouche blanche (bemisia tabaci).

[2] Classe: Healthy
    Question: Qu'est-ce qui cause ces symptômes ?
    Réponse: ce plant de manioc ne présente **aucune maladie**. les feuilles montrent une coloration et une forme normales. veillez à maintenir de bonnes pratiques culturales pour préserver cette santé.

[3] Classe: CMD
    Question: Que dois-je faire pour traiter cette maladie ?
    Réponse: actions à prendre pour la cmd : évaluer la sévérité et agir en conséquence. pour protéger vos autres plants : planter en début de saison des pluies. inspecter visuellement avant plantation. utiliser des boutures de sources certifiées.

[4] Classe: CBB
    Question: Quels sont les signes de maladie que vous observez ?
    Réponse: j'observe les symptômes suivants caractéristiques de striure brune 

In [10]:
# ============================================================
# CELL 9: CONCLUSION
# ============================================================
print("\n" + "="*60)
print("CONCLUSION - ÉVALUATION DATASET DE TEST")
print("="*60)

# Performance globale
print(f"\n🎯 Performance sur dataset de test:")
print(f"   Accuracy: {accuracy:.2f}%")
print(f"   Échantillons: {len(results):,} images")

# Évaluation de la performance
if accuracy >= 85:
    print(f"\n🏆 Performance excellente! (≥85%)")
elif accuracy >= 75:
    print(f"\n🎉 Performance très bonne! (75-85%)")
elif accuracy >= 65:
    print(f"\n✅ Performance acceptable (65-75%)")
elif accuracy >= 55:
    print(f"\n⚠️ Performance modérée (55-65%)")
else:
    print(f"\n⚠️ Performance faible (<55%)")

# Comparaison indicative avec Stage 1 (datasets différents)
stage1_global = STAGE1_RESULTS.get('global', 0)
if stage1_global > 0:
    retention = accuracy - stage1_global
    print(f"\n📊 Comparaison indicative avec Stage 1:")
    print(f"   ⚠️ Note: Datasets différents (validation vs test)")
    if abs(retention) <= 5:
        print(f"   📈 Cohérence: Δ{retention:+.1f}% (très proche)")
    elif retention > 0:
        print(f"   📈 Amélioration: +{retention:.1f}%")
    else:
        print(f"   📉 Écart: {retention:.1f}%")

# Matrice de confusion simplifiée
print("\n📊 Matrice de confusion simplifiée:")
confusion = {}
for cls in CLASS_NAMES:
    confusion[cls] = Counter(r['predicted'] for r in results if r['actual'] == cls)

for actual in CLASS_NAMES:
    predictions = confusion[actual]
    if predictions:
        main_pred = predictions.most_common(1)[0]
        print(f"  {actual} → {main_pred[0]} ({main_pred[1]}x)")

print(f"\n📂 Informations techniques:")
print(f"   Checkpoint: {CHECKPOINT_PATH}")
print(f"   Dataset: Test officiel (sample_submission_file.csv)")
print(f"   Images: /data/cassava/test/0/")


CONCLUSION - ÉVALUATION DATASET DE TEST

🎯 Performance sur dataset de test:
   Accuracy: 24.00%
   Échantillons: 100 images

⚠️ Performance faible (<55%)

📊 Comparaison indicative avec Stage 1:
   ⚠️ Note: Datasets différents (validation vs test)
   📉 Écart: -56.5%

📊 Matrice de confusion simplifiée:
  CBB → CMD (9x)
  CBSD → CBSD (7x)
  CGM → CMD (3x)
  CMD → CMD (15x)
  Healthy → CMD (15x)

📂 Informations techniques:
   Checkpoint: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v5_unified_final/final_model
   Dataset: Test officiel (sample_submission_file.csv)
   Images: /data/cassava/test/0/


In [11]:
# ============================================================
# CELL 10: INVESTIGATION - HEALTHY CLASS CONFUSION
# ============================================================
print("="*60)
print("INVESTIGATION: HEALTHY CLASS CONFUSION")
print("="*60)

# Filtrer les samples Healthy
healthy_samples = [r for r in results if r['actual'] == 'Healthy']
print(f"\nSamples Healthy dans l'évaluation: {len(healthy_samples)}")

# Distribution des prédictions pour Healthy
healthy_predictions = Counter(r['predicted'] for r in healthy_samples)
print("\n📊 Quand la classe réelle est Healthy, le modèle prédit:")
for pred, count in healthy_predictions.most_common():
    pct = count / len(healthy_samples) * 100
    status = "✅" if pred == "Healthy" else "❌"
    print(f"  {status} {pred}: {count} ({pct:.1f}%)")

# Matrice de confusion complète
print("\n📊 Matrice de confusion complète:")
print(f"{'Actual':<10} | " + " | ".join(f"{c:<8}" for c in CLASS_NAMES + ["Unknown"]))
print("-" * 75)
for actual in CLASS_NAMES:
    row_samples = [r for r in results if r['actual'] == actual]
    if row_samples:
        counts = []
        for pred in CLASS_NAMES + ["Unknown"]:
            count = sum(1 for r in row_samples if r['predicted'] == pred)
            counts.append(f"{count:<8}")
        print(f"{actual:<10} | " + " | ".join(counts))

# Exemples de Healthy mal classifiés
print("\n" + "="*60)
print("EXEMPLES: Healthy → Maladie (erreurs)")
print("="*60)

healthy_errors = [r for r in healthy_samples if not r['correct']]
for i, sample in enumerate(healthy_errors[:5]):
    print(f"\n[{i+1}] Healthy → {sample['predicted']}")
    print(f"    Réponse: {sample['response'][:250]}")
    if len(sample['response']) > 250:
        print("...")

# Analyse des patterns d'erreur
print("\n" + "="*60)
print("ANALYSE DES PATTERNS D'ERREUR")
print("="*60)

# Quelles maladies "volent" les samples Healthy?
if healthy_errors:
    error_dist = Counter(r['predicted'] for r in healthy_errors)
    print("\nLes samples Healthy sont principalement confondus avec:")
    for disease, count in error_dist.most_common():
        pct = count / len(healthy_errors) * 100
        print(f"  → {disease}: {count} erreurs ({pct:.1f}%)")

    # Recommandation
    top_confusion = error_dist.most_common(1)[0][0]
    print(f"\n💡 Recommandation: Le modèle confond Healthy avec {top_confusion}.")
    print("   Causes possibles:")
    print("   - Sous-représentation de Healthy dans les données d'entraînement")
    print("   - Biais vers la détection de maladies (format 'Diagnostic:')")
    print("   - Similarité visuelle entre feuilles saines et certains stades précoces")

INVESTIGATION: HEALTHY CLASS CONFUSION

Samples Healthy dans l'évaluation: 24

📊 Quand la classe réelle est Healthy, le modèle prédit:
  ❌ CMD: 15 (62.5%)
  ❌ CBB: 4 (16.7%)
  ❌ CBSD: 3 (12.5%)
  ❌ CGM: 2 (8.3%)

📊 Matrice de confusion complète:
Actual     | CBB      | CBSD     | CGM      | CMD      | Healthy  | Unknown 
---------------------------------------------------------------------------
CBB        | 0        | 4        | 4        | 9        | 5        | 0       
CBSD       | 3        | 7        | 4        | 6        | 1        | 0       
CGM        | 3        | 2        | 2        | 3        | 1        | 1       
CMD        | 0        | 4        | 2        | 15       | 0        | 0       
Healthy    | 4        | 3        | 2        | 15       | 0        | 0       

EXEMPLES: Healthy → Maladie (erreurs)

[1] Healthy → CBSD
    Réponse: diagnostic : cassava brown streak disease (cbsd). cette feuille de manioc présente des symptômes caractéristiques de cette condition.

[2] Healt